In [1]:
import polars as pl
import warnings
import os 

warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)
os.environ['POLARS_WARN_UNSTABLE'] = '0'

In [2]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

polars.config.Config

In [3]:
# Load all data files using Polars
print("Loading datasets with Polars...")
# Additional data sources
credit_card_balance = pl.read_csv('../data/credit_card_balance.csv')
print(f"Credit card balance data shape: {credit_card_balance.shape}")

Loading datasets with Polars...
Credit card balance data shape: (3840312, 23)


In [4]:
# === CREDIT CARD BALANCE EDA ===
print("=== CREDIT CARD BALANCE DETAILED ANALYSIS ===")

print(f"Dataset shape: {credit_card_balance.shape}")
print(f"Columns: {list(credit_card_balance.columns)}")

# Basic statistics
print(f"\nBasic Statistics:")
print(f"Unique loans: {credit_card_balance['SK_ID_CURR'].n_unique():,}")
print(f"Unique previous loans: {credit_card_balance['SK_ID_PREV'].n_unique():,}")
print(f"Total monthly records: {credit_card_balance.shape[0]:,}")
print(f"Average records per customer: {credit_card_balance.shape[0] / credit_card_balance['SK_ID_CURR'].n_unique():.1f}")

# Sample data
print(f"\nFirst 10 rows:")
print(credit_card_balance.head(10))

# Missing values analysis
print(f"\nMissing Values Analysis:")
missing_analysis = []
for col in credit_card_balance.columns:
    null_count = credit_card_balance[col].null_count()
    null_pct = (null_count / credit_card_balance.shape[0]) * 100
    missing_analysis.append({'column': col, 'null_count': null_count, 'null_percentage':
null_pct})

missing_df = pl.DataFrame(missing_analysis).sort('null_percentage', descending=True)
print(missing_df.filter(pl.col('null_count') > 0))


=== CREDIT CARD BALANCE DETAILED ANALYSIS ===
Dataset shape: (3840312, 23)
Columns: ['SK_ID_PREV', 'SK_ID_CURR', 'MONTHS_BALANCE', 'AMT_BALANCE', 'AMT_CREDIT_LIMIT_ACTUAL', 'AMT_DRAWINGS_ATM_CURRENT', 'AMT_DRAWINGS_CURRENT', 'AMT_DRAWINGS_OTHER_CURRENT', 'AMT_DRAWINGS_POS_CURRENT', 'AMT_INST_MIN_REGULARITY', 'AMT_PAYMENT_CURRENT', 'AMT_PAYMENT_TOTAL_CURRENT', 'AMT_RECEIVABLE_PRINCIPAL', 'AMT_RECIVABLE', 'AMT_TOTAL_RECEIVABLE', 'CNT_DRAWINGS_ATM_CURRENT', 'CNT_DRAWINGS_CURRENT', 'CNT_DRAWINGS_OTHER_CURRENT', 'CNT_DRAWINGS_POS_CURRENT', 'CNT_INSTALMENT_MATURE_CUM', 'NAME_CONTRACT_STATUS', 'SK_DPD', 'SK_DPD_DEF']

Basic Statistics:
Unique loans: 103,558
Unique previous loans: 104,307
Total monthly records: 3,840,312
Average records per customer: 37.1

First 10 rows:
shape: (10, 23)
┌────────────┬────────────┬────────────────┬─────────────┬─────────────────────────┬──────────────────────────┬──────────────────────┬────────────────────────────┬──────────────────────────┬────────────────────

In [5]:

# === KEY BEHAVIORAL PATTERNS ANALYSIS ===
print("\n=== CREDIT CARD BEHAVIORAL PATTERNS ===")

# 1. Contract Status Analysis
print("1. CONTRACT STATUS DISTRIBUTION:")
status_counts = credit_card_balance['NAME_CONTRACT_STATUS'].value_counts().sort('count',descending=True)
print(status_counts)

# 2. Payment vs Balance Analysis
print("\n2. PAYMENT BEHAVIOR ANALYSIS:")
payment_analysis = credit_card_balance.select([
    pl.col('AMT_BALANCE').min().alias('min_balance'),
    pl.col('AMT_BALANCE').max().alias('max_balance'),
    pl.col('AMT_BALANCE').mean().alias('avg_balance'),
    pl.col('AMT_PAYMENT_CURRENT').min().alias('min_payment'),
    pl.col('AMT_PAYMENT_CURRENT').max().alias('max_payment'),
    pl.col('AMT_PAYMENT_CURRENT').mean().alias('avg_payment'),
    (pl.col('AMT_PAYMENT_CURRENT') == 0).sum().alias('zero_payment_months'),
    (pl.col('AMT_BALANCE') >
pl.col('AMT_PAYMENT_CURRENT')).sum().alias('underpayment_months')
])
print(payment_analysis)

# 3. Credit Utilization Patterns
print("\n3. CREDIT UTILIZATION ANALYSIS:")
utilization_analysis = credit_card_balance.with_columns([
    (pl.col('AMT_BALANCE') /
pl.col('AMT_CREDIT_LIMIT_ACTUAL')).alias('utilization_ratio')
]).select([
    pl.col('utilization_ratio').min().alias('min_utilization'),
    pl.col('utilization_ratio').max().alias('max_utilization'),
    pl.col('utilization_ratio').mean().alias('avg_utilization'),
    pl.col('utilization_ratio').median().alias('median_utilization'),
    (pl.col('utilization_ratio') > 0.8).sum().alias('high_utilization_months'),
    (pl.col('utilization_ratio') > 1.0).sum().alias('over_limit_months')
])
print(utilization_analysis)

# 4. DPD (Days Past Due) Analysis
print("\n4. DELINQUENCY ANALYSIS:")
dpd_analysis = credit_card_balance.select([
    pl.col('SK_DPD').mean().alias('avg_dpd'),
    pl.col('SK_DPD').max().alias('max_dpd'),
    (pl.col('SK_DPD') == 0).sum().alias('on_time_months'),
    (pl.col('SK_DPD').is_between(1, 30)).sum().alias('dpd_1_30_months'),
    (pl.col('SK_DPD').is_between(31, 60)).sum().alias('dpd_31_60_months'),
    (pl.col('SK_DPD') > 60).sum().alias('dpd_60plus_months'),
    (pl.col('SK_DPD') > 0).sum().alias('any_dpd_months')
])
print(dpd_analysis)

# 5. Spending Pattern Analysis
print("\n5. SPENDING PATTERNS:")
spending_analysis = credit_card_balance.select([
    pl.col('AMT_DRAWINGS_CURRENT').mean().alias('avg_monthly_spending'),
    pl.col('AMT_DRAWINGS_ATM_CURRENT').mean().alias('avg_atm_withdrawals'),
    pl.col('AMT_DRAWINGS_POS_CURRENT').mean().alias('avg_pos_purchases'),
    pl.col('CNT_DRAWINGS_CURRENT').mean().alias('avg_transactions_per_month'),
    pl.col('CNT_DRAWINGS_ATM_CURRENT').mean().alias('avg_atm_transactions'),
    (pl.col('AMT_DRAWINGS_CURRENT') == 0).sum().alias('inactive_months')
])
print(spending_analysis)



=== CREDIT CARD BEHAVIORAL PATTERNS ===
1. CONTRACT STATUS DISTRIBUTION:
shape: (7, 2)
┌──────────────────────┬─────────┐
│ NAME_CONTRACT_STATUS ┆ count   │
│ ---                  ┆ ---     │
│ str                  ┆ u32     │
╞══════════════════════╪═════════╡
│ Active               ┆ 3698436 │
│ Completed            ┆ 128918  │
│ Signed               ┆ 11058   │
│ Demand               ┆ 1365    │
│ Sent proposal        ┆ 513     │
│ Refused              ┆ 17      │
│ Approved             ┆ 5       │
└──────────────────────┴─────────┘

2. PAYMENT BEHAVIOR ANALYSIS:
shape: (1, 8)
┌─────────────┬─────────────┬──────────────┬─────────────┬─────────────┬──────────────┬─────────────────────┬─────────────────────┐
│ min_balance ┆ max_balance ┆ avg_balance  ┆ min_payment ┆ max_payment ┆ avg_payment  ┆ zero_payment_months ┆ underpayment_months │
│ ---         ┆ ---         ┆ ---          ┆ ---         ┆ ---         ┆ ---          ┆ ---                 ┆ ---                 │
│ f64         ┆ 

In [6]:


# === CREDIT CARD FEATURE ENGINEERING ===
print("=== CREDIT CARD FEATURE ENGINEERING ===")

# Create comprehensive credit card features at SK_ID_CURR level
credit_card_features = credit_card_balance.group_by('SK_ID_CURR').agg([
    # === BASIC UTILIZATION FEATURES ===
    pl.count().alias('CC_MONTHS_COUNT'),
    pl.col('SK_ID_PREV').n_unique().alias('CC_CARDS_COUNT'),

    # Credit utilization patterns
    (pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')).mean().alias('CC_UTILIZATION_MEAN'),
    (pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')).max().alias('CC_UTILIZATION_MAX'),
    (pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')).std().alias('CC_UTILIZATION_STD'),

    # High utilization risk indicators
    ((pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')) > 0.8).sum().alias('CC_HIGH_UTIL_MONTHS'),
    ((pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')) > 1.0).sum().alias('CC_OVER_LIMIT_MONTHS'),
    (((pl.col('AMT_BALANCE') / pl.col('AMT_CREDIT_LIMIT_ACTUAL')) > 0.8).sum() / pl.count()).alias('CC_HIGH_UTIL_RATIO'),

    # === PAYMENT BEHAVIOR FEATURES ===
    # Payment-to-balance ratios (key credit risk indicator)
    (pl.col('AMT_PAYMENT_CURRENT') / pl.col('AMT_BALANCE')).mean().alias('CC_PAYMENT_RATIO_MEAN'),
    (pl.col('AMT_PAYMENT_CURRENT') / pl.col('AMT_BALANCE')).min().alias('CC_PAYMENT_RATIO_MIN'),

    # Minimum payment adherence
    (pl.col('AMT_PAYMENT_CURRENT') / pl.col('AMT_INST_MIN_REGULARITY')).mean().alias('CC_MIN_PAYMENT_RATIO'),
    (pl.col('AMT_PAYMENT_CURRENT') < pl.col('AMT_INST_MIN_REGULARITY')).sum().alias('CC_UNDERPAY_MONTHS'),
    ((pl.col('AMT_PAYMENT_CURRENT') < pl.col('AMT_INST_MIN_REGULARITY')).sum() / pl.count()).alias('CC_UNDERPAY_RATIO'),

    # Zero payment months (major risk signal)
    (pl.col('AMT_PAYMENT_CURRENT') == 0).sum().alias('CC_NO_PAYMENT_MONTHS'),
    ((pl.col('AMT_PAYMENT_CURRENT') == 0).sum() / pl.count()).alias('CC_NO_PAYMENT_RATIO'),

    # === DPD (DELINQUENCY) FEATURES ===
    pl.col('SK_DPD').sum().alias('CC_DPD_TOTAL'),
    pl.col('SK_DPD').mean().alias('CC_DPD_MEAN'),
    pl.col('SK_DPD').max().alias('CC_DPD_MAX'),

    # DPD frequency patterns
    (pl.col('SK_DPD') > 0).sum().alias('CC_DPD_MONTHS'),
    (pl.col('SK_DPD').is_between(1, 30)).sum().alias('CC_DPD_1_30_MONTHS'),
    (pl.col('SK_DPD').is_between(31, 60)).sum().alias('CC_DPD_31_60_MONTHS'),
    (pl.col('SK_DPD') > 60).sum().alias('CC_DPD_60PLUS_MONTHS'),

    # DPD ratios (key risk indicators)
    ((pl.col('SK_DPD') > 0).sum() / pl.count()).alias('CC_DPD_RATIO'),
    ((pl.col('SK_DPD') > 60).sum() / pl.count()).alias('CC_SEVERE_DPD_RATIO'),

    # === SPENDING BEHAVIOR FEATURES ===
    # Total spending patterns
    pl.col('AMT_DRAWINGS_CURRENT').sum().alias('CC_TOTAL_DRAWINGS'),
    pl.col('AMT_DRAWINGS_CURRENT').mean().alias('CC_DRAWINGS_MEAN'),
    pl.col('AMT_DRAWINGS_CURRENT').std().alias('CC_DRAWINGS_STD'),

    # Spending breakdown by type
    pl.col('AMT_DRAWINGS_ATM_CURRENT').sum().alias('CC_ATM_TOTAL'),
    pl.col('AMT_DRAWINGS_POS_CURRENT').sum().alias('CC_POS_TOTAL'),
    pl.col('AMT_DRAWINGS_OTHER_CURRENT').sum().alias('CC_OTHER_TOTAL'),

    # Spending ratios (behavioral indicators)
    (pl.col('AMT_DRAWINGS_ATM_CURRENT').sum() /
pl.col('AMT_DRAWINGS_CURRENT').sum()).alias('CC_ATM_RATIO'),
    (pl.col('AMT_DRAWINGS_POS_CURRENT').sum() /
pl.col('AMT_DRAWINGS_CURRENT').sum()).alias('CC_POS_RATIO'),

    # Transaction frequency
    pl.col('CNT_DRAWINGS_CURRENT').mean().alias('CC_TXN_FREQ_MEAN'),
    pl.col('CNT_DRAWINGS_ATM_CURRENT').sum().alias('CC_ATM_TXN_COUNT'),
    pl.col('CNT_DRAWINGS_POS_CURRENT').sum().alias('CC_POS_TXN_COUNT'),

    # === TEMPORAL PATTERNS ===
    pl.col('MONTHS_BALANCE').min().alias('CC_EARLIEST_MONTH'),
    pl.col('MONTHS_BALANCE').max().alias('CC_LATEST_MONTH'),

    # Activity patterns
    (pl.col('AMT_DRAWINGS_CURRENT') == 0).sum().alias('CC_INACTIVE_MONTHS'),
    ((pl.col('AMT_DRAWINGS_CURRENT') == 0).sum() /
pl.count()).alias('CC_INACTIVE_RATIO'),

    # === LIMIT MANAGEMENT ===
    pl.col('AMT_CREDIT_LIMIT_ACTUAL').mean().alias('CC_LIMIT_MEAN'),
    pl.col('AMT_CREDIT_LIMIT_ACTUAL').max().alias('CC_LIMIT_MAX'),

    # Balance management
    pl.col('AMT_BALANCE').mean().alias('CC_BALANCE_MEAN'),
    pl.col('AMT_BALANCE').max().alias('CC_BALANCE_MAX'),
    (pl.col('AMT_BALANCE').max() /
pl.col('AMT_CREDIT_LIMIT_ACTUAL').max()).alias('CC_MAX_UTIL_EVER'),
])

print(f"Credit card features shape: {credit_card_features.shape}")
print(f"Features created: {len(credit_card_features.columns) - 1}")  # -1 for SK_ID_CURR

# Display feature summary
print(f"\nCreated credit card features:")
feature_categories = {
    'Utilization': [col for col in credit_card_features.columns if 'UTIL' in col],
    'Payment Behavior': [col for col in credit_card_features.columns if 'PAYMENT' in col
or 'PAY' in col],
    'Delinquency': [col for col in credit_card_features.columns if 'DPD' in col],
    'Spending': [col for col in credit_card_features.columns if 'DRAWING' in col or
'TXN' in col or 'ATM' in col or 'POS' in col],
    'Account Management': [col for col in credit_card_features.columns if 'BALANCE' in
col or 'LIMIT' in col or 'COUNT' in col]
}

for category, features in feature_categories.items():
    print(f"\n📊 {category} Features ({len(features)}):")
    for feat in features[:5]:  # Show first 5 in each category
        print(f"   • {feat}")
    if len(features) > 5:
        print(f"   ... and {len(features) - 5} more")

=== CREDIT CARD FEATURE ENGINEERING ===
Credit card features shape: (103558, 45)
Features created: 44

Created credit card features:

📊 Utilization Features (6):
   • CC_UTILIZATION_MEAN
   • CC_UTILIZATION_MAX
   • CC_UTILIZATION_STD
   • CC_HIGH_UTIL_MONTHS
   • CC_HIGH_UTIL_RATIO
   ... and 1 more

📊 Payment Behavior Features (7):
   • CC_PAYMENT_RATIO_MEAN
   • CC_PAYMENT_RATIO_MIN
   • CC_MIN_PAYMENT_RATIO
   • CC_UNDERPAY_MONTHS
   • CC_UNDERPAY_RATIO
   ... and 2 more

📊 Delinquency Features (9):
   • CC_DPD_TOTAL
   • CC_DPD_MEAN
   • CC_DPD_MAX
   • CC_DPD_MONTHS
   • CC_DPD_1_30_MONTHS
   ... and 4 more

📊 Spending Features (10):
   • CC_TOTAL_DRAWINGS
   • CC_DRAWINGS_MEAN
   • CC_DRAWINGS_STD
   • CC_ATM_TOTAL
   • CC_POS_TOTAL
   ... and 5 more

📊 Account Management Features (9):
   • CC_MONTHS_COUNT
   • CC_CARDS_COUNT
   • CC_OVER_LIMIT_MONTHS
   • CC_ATM_TXN_COUNT
   • CC_POS_TXN_COUNT
   ... and 4 more


In [7]:
# === FEATURE QUALITY VALIDATION ===
print("\n=== CREDIT CARD FEATURE QUALITY VALIDATION ===")

# Check feature completeness and data quality
print(f"Total customers in credit card features: {credit_card_features.shape[0]:,}")
print(f"Total features created: {len(credit_card_features.columns) - 1}")

# Missing value analysis for created features
print("\nMissing values in engineered features:")
missing_features = []
for col in credit_card_features.columns:
    if col != 'SK_ID_CURR':
        null_count = credit_card_features[col].null_count()
        if null_count > 0:
            missing_features.append({
                'feature': col,
                'null_count': null_count,
                'null_pct': (null_count / credit_card_features.shape[0]) * 100
            })

if missing_features:
    missing_df = pl.DataFrame(missing_features).sort('null_pct', descending=True)
    print(missing_df)
else:
    print("✅ No missing values in engineered features!")

# Infinite/extreme value checks
print("\nChecking for infinite or extreme values:")
extreme_features = []
for col in credit_card_features.columns:
    if col != 'SK_ID_CURR' and credit_card_features[col].dtype in [pl.Float64,
pl.Float32]:
        col_stats = credit_card_features.select([
            pl.col(col).min().alias('min'),
            pl.col(col).max().alias('max'),
            pl.col(col).is_infinite().sum().alias('inf_count')
        ]).row(0)

        if col_stats[2] > 0 or col_stats[0] < -1e10 or col_stats[1] > 1e10:
            extreme_features.append({
                'feature': col,
                'min': col_stats[0],
                'max': col_stats[1],
                'inf_count': col_stats[2]
            })

if extreme_features:
    print("⚠️  Features with extreme/infinite values:")
    for feat in extreme_features:
        print(f"   {feat['feature']}: min={feat['min']:.2e}, max={feat['max']:.2e}, inf_count={feat['inf_count']}")
else:
    print("✅ No extreme or infinite values detected!")

# Feature distribution summary
print("\nKey feature distributions (top risk indicators):")
risk_features = ['CC_DPD_RATIO', 'CC_SEVERE_DPD_RATIO', 'CC_NO_PAYMENT_RATIO',
                'CC_HIGH_UTIL_RATIO', 'CC_UNDERPAY_RATIO']

for feat in risk_features:
    if feat in credit_card_features.columns:
        stats = credit_card_features.select([
            pl.col(feat).mean().alias('mean'),
            pl.col(feat).std().alias('std'),
            pl.col(feat).quantile(0.5).alias('median'),
            pl.col(feat).quantile(0.95).alias('p95')
        ]).row(0)
        print(f"📊 {feat}: mean={stats[0]:.3f}, std={stats[1]:.3f}, median={stats[2]:.3f}, p95={stats[3]:.3f}")


=== CREDIT CARD FEATURE QUALITY VALIDATION ===
Total customers in credit card features: 103,558
Total features created: 44

Missing values in engineered features:
shape: (5, 3)
┌───────────────────────┬────────────┬───────────┐
│ feature               ┆ null_count ┆ null_pct  │
│ ---                   ┆ ---        ┆ ---       │
│ str                   ┆ i64        ┆ f64       │
╞═══════════════════════╪════════════╪═══════════╡
│ CC_PAYMENT_RATIO_MEAN ┆ 31438      ┆ 30.357867 │
│ CC_PAYMENT_RATIO_MIN  ┆ 31438      ┆ 30.357867 │
│ CC_MIN_PAYMENT_RATIO  ┆ 31438      ┆ 30.357867 │
│ CC_UTILIZATION_STD    ┆ 692        ┆ 0.668225  │
│ CC_DRAWINGS_STD       ┆ 692        ┆ 0.668225  │
└───────────────────────┴────────────┴───────────┘

Checking for infinite or extreme values:
⚠️  Features with extreme/infinite values:
   CC_UTILIZATION_MEAN: min=-2.56e-03, max=inf, inf_count=230
   CC_UTILIZATION_MAX: min=0.00e+00, max=inf, inf_count=768
   CC_PAYMENT_RATIO_MEAN: min=-8.07e+02, max=inf, inf_

In [8]:
# === SAVE CREDIT CARD FEATURES ===
print("\n=== SAVING CREDIT CARD FEATURES ===")

# Clean infinite values before saving
credit_card_features_clean = credit_card_features.with_columns([
    pl.when(pl.col(col).is_infinite())
    .then(None)
    .otherwise(pl.col(col))
    .alias(col)
    for col in credit_card_features.columns if col != 'SK_ID_CURR'
])

# Save to CSV for integration with main model
output_path = '../data/credit_card_features.csv'
credit_card_features_clean.write_csv(output_path)
print(f"✅ Credit card features saved to: {output_path}")
print(f"Shape: {credit_card_features_clean.shape}")

# Feature summary for documentation
print(f"\n📋 CREDIT CARD FEATURE SUMMARY:")
print(f"Total customers with credit card history: {credit_card_features_clean.shape[0]:,}")
print(f"Total features engineered: {len(credit_card_features_clean.columns) - 1}")

# Feature importance preview (based on business logic)
high_importance_features = [
    'CC_DPD_RATIO', 'CC_SEVERE_DPD_RATIO', 'CC_NO_PAYMENT_RATIO',
    'CC_HIGH_UTIL_RATIO', 'CC_UNDERPAY_RATIO', 'CC_UTILIZATION_MEAN',
    'CC_PAYMENT_RATIO_MEAN', 'CC_MAX_UTIL_EVER'
]

print(f"\n🎯 HIGH-IMPACT RISK FEATURES (ready for model integration):")

for feat in high_importance_features:
    if feat in credit_card_features_clean.columns:
        print(f"   • {feat}")

print(f"\n💡 INTEGRATION INSTRUCTIONS:")
print(f"1. Merge with application_train on SK_ID_CURR")
print(f"2. Handle missing values (customers without credit card history)")
print(f"3. Add to existing baseline + POS + bureau feature set")
print(f"4. Expected AUC improvement: +0.005 to +0.015 based on credit behavior patterns")


=== SAVING CREDIT CARD FEATURES ===
✅ Credit card features saved to: ../data/credit_card_features.csv
Shape: (103558, 45)

📋 CREDIT CARD FEATURE SUMMARY:
Total customers with credit card history: 103,558
Total features engineered: 44

🎯 HIGH-IMPACT RISK FEATURES (ready for model integration):
   • CC_DPD_RATIO
   • CC_SEVERE_DPD_RATIO
   • CC_NO_PAYMENT_RATIO
   • CC_HIGH_UTIL_RATIO
   • CC_UNDERPAY_RATIO
   • CC_UTILIZATION_MEAN
   • CC_PAYMENT_RATIO_MEAN
   • CC_MAX_UTIL_EVER

💡 INTEGRATION INSTRUCTIONS:
1. Merge with application_train on SK_ID_CURR
2. Handle missing values (customers without credit card history)
3. Add to existing baseline + POS + bureau feature set
4. Expected AUC improvement: +0.005 to +0.015 based on credit behavior patterns
